In [49]:
'Import libraries'
import numpy as np
import pandas as pd
import xarray as xr
from numba import jit
import dask
import netCDF4
import h5netcdf
from scipy.stats import mannwhitneyu
import sklearn

In [3]:
north = xr.open_dataset('/scratch/jbos/sst_cubes/sst_cube_North.nc',engine="netcdf4")
south = xr.open_dataset('/scratch/jbos/sst_cubes/sst_cube_South.nc',engine="netcdf4")

In [4]:
sst_south = south['sst']
sst_north = north['sst']

In [5]:
def mean_sst(row,sst):
    a = sst.sel(y = row['Latitude'], x = row['Longitude'] , method= "nearest")
    a = a.values.mean()
    if (a > 0):
        return(a)
    else:
        sst_i = sst.where(sst['y'] < row['Latitude']+ 0.001, drop=True)
        sst_i = sst_i.where(sst_i['y'] > row['Latitude']- 0.001, drop=True)
        sst_i = sst_i.where(sst_i['x'] < row['Longitude']+ 0.001, drop=True)
        sst_i = sst_i.where(sst_i['x'] > row['Longitude']-0.001, drop=True)
        b = sst_i.mean()
        b = b.values.item()
        if (b > 0):
            return(b)
        else:        
            sst_i = sst.where(sst['y'] < row['Latitude']+ 0.002, drop=True)
            sst_i = sst_i.where(sst_i['y'] > row['Latitude']- 0.002, drop=True)
            sst_i = sst_i.where(sst_i['x'] < row['Longitude']+ 0.002, drop=True)
            sst_i = sst_i.where(sst_i['x'] > row['Longitude']-0.002, drop=True)
            c = sst_i.mean()
            return(c.values.item())

'Filepath for input logger data'
path = '/home/jbos/metadat/'

'Filepath for output data'
out_path = '/home/jbos'

In [6]:
def q99_sst(row,sst):
    a = sst.sel(y = row['Latitude'], x = row['Longitude'] , method= "nearest")
    a = np.percentile(a, 99)
    if (a > 0):
        return(a)
    else:
        sst_i = sst.where(sst['y'] < row['Latitude']+ 0.001, drop=True)
        sst_i = sst_i.where(sst_i['y'] > row['Latitude']- 0.001, drop=True)
        sst_i = sst_i.where(sst_i['x'] < row['Longitude']+ 0.001, drop=True)
        sst_i = sst_i.where(sst_i['x'] > row['Longitude']-0.001, drop=True)
        b = sst_i.mean()
        b = np.percentile(b, 99)
        if (b > 0):
            return(b)
        else:        
            sst_i = sst.where(sst['y'] < row['Latitude']+ 0.002, drop=True)
            sst_i = sst_i.where(sst_i['y'] > row['Latitude']- 0.002, drop=True)
            sst_i = sst_i.where(sst_i['x'] < row['Longitude']+ 0.002, drop=True)
            sst_i = sst_i.where(sst_i['x'] > row['Longitude']-0.002, drop=True)
            c = sst_i.mean()
            return(np.percentile(c, 99))

'Filepath for input logger data'
path = '/home/jbos/metadat/'

'Filepath for output data'
out_path = '/home/jbos/'

In [7]:
#Read in metadata
metadat = pd.read_csv('/home/jbos/metadat_zooxantelas.csv')

In [8]:
metadat_north = metadat.loc[metadat['Latitude']>(-15)]
metadat_south = metadat.loc[metadat['Latitude']<(-15)]

In [9]:
metadat_north['mean_sst'] = metadat_north.apply(mean_sst,axis=1,sst=sst_north)
metadat_north['q99_sst'] = metadat_north.apply(q99_sst,axis=1,sst=sst_north)

In [10]:
metadat_south['mean_sst'] = metadat_south.apply(mean_sst,axis=1,sst=sst_south)
metadat_south['q99_sst'] = metadat_south.apply(q99_sst,axis=1,sst=sst_south)

In [11]:
db_means=metadat_north.loc[metadat['Acropora_spp']=='DB']['mean_sst']

In [12]:
da_means=metadat_north.loc[metadat['Acropora_spp']=='DA']['mean_sst']

In [13]:
mannwhitneyu(db_means,da_means)

MannwhitneyuResult(statistic=np.float64(5235.0), pvalue=np.float64(3.2602484697467584e-09))

In [14]:
db_q99=metadat_north.loc[metadat['Acropora_spp']=='DB']['q99_sst']

In [15]:
da_q99=metadat_north.loc[metadat['Acropora_spp']=='DA']['q99_sst']

In [16]:
db_q99.median()

np.float64(30.48890047073364)

In [17]:
da_q99.median()

np.float64(30.43989688873291)

In [18]:
mannwhitneyu(db_q99,da_q99)

MannwhitneyuResult(statistic=np.float64(5571.0), pvalue=np.float64(2.319659333046598e-12))

In [20]:
bleaching_meta = pd.read_csv('/home/jbos/Moz_reads/Bleaching_Jaelyn.csv')

In [23]:
bleaching_meta.columns

Index(['Numero_do_tubo', 'Latitude', 'Longitude', 'Profundidade_metros', 'Dia',
       'Mes', 'Ano', 'Bleaching', 'Death'],
      dtype='str')

In [26]:
def format_id(id_num):
    return f"ACR_{id_num:03d}"

In [25]:
bleaching_meta['Bleaching'].value_counts()

Bleaching
0.0    125
1.0     49
Name: count, dtype: int64

In [43]:
bleaching_meta.loc[53,'Numero_do_tubo'] = '613'
bleaching_meta.loc[54,'Numero_do_tubo'] = '616'

In [44]:
bleaching_meta['IND'] = pd.to_numeric(bleaching_meta['Numero_do_tubo']).apply(format_id)

In [45]:
meta_bleaching=pd.merge(metadat_north,bleaching_meta,on='IND',how='left')

In [46]:
meta_bleaching=meta_bleaching[

,Unnamed: 0,IND,Latitude_x,Longitude_x,Profundidade_metros_x,Dia_x,Mes_x,Ano_x,Cladocopium_leituras,Durusdinium_leituras,...,q99_sst,Numero_do_tubo,Latitude_y,Longitude_y,Profundidade_metros_y,Dia_y,Mes_y,Ano_y,Bleaching,Death
0,1,ACR_006,-12.912900,40.499890,2.4,2,6,2023,872564,39959,...,30.487585,6,-12.912900,40.499890,2.4,2,6,2023,NaN,NaN
1,2,ACR_010,-12.919217,40.518081,8.0,12,6,2023,2403654,33899,...,30.503295,10,-12.919217,40.518081,8.0,12,6,2023,NaN,NaN
2,3,ACR_011,-12.912900,40.499890,2.1,2,6,2023,3375691,96794,...,30.487585,11,-12.912900,40.499890,2.1,2,6,2023,NaN,NaN
3,4,ACR_018,-12.912900,40.499890,1.8,2,6,2023,1115994,56935,...,30.487585,18,-12.912900,40.499890,1.8,2,6,2023,NaN,NaN
4,5,ACR_019,-12.912900,40.499890,1.9,2,6,2023,493467,17547,...,30.487585,19,-12.912900,40.499890,1.9,2,6,2023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,231,ACR_723,-12.905278,40.498611,2.9,3,7,2024,1080713,100444,...,30.510966,723,-12.905278,40.498611,2.9,3,7,2024,1.0,0.0
163,232,ACR_726,-12.905278,40.498611,3.0,3,7,2024,1039092,73789,...,30.510966,726,-12.905278,40.498611,3.0,3,7,2024,1.0,1.0
164,233,ACR_729,-12.905278,40.498611,2.6,3,7,2024,671937,32905,...,30.510966,729,-12.905278,40.498611,2.6,3,7,2024,1.0,0.0
165,234,ACR_801,-12.915556,40.500278,1.8,8,7,2024,205679,12452,...,28.263662,801,-12.915556,40.500278,1.8,8,7,2024,0.0,0.0


In [60]:
meta_bleaching['Bleaching']
meta_bleaching24 = meta_bleaching.loc[-pd.isna(meta_bleaching['Bleaching'])]

In [61]:
meta_bleaching24

,Unnamed: 0,IND,Latitude_x,Longitude_x,Profundidade_metros_x,Dia_x,Mes_x,Ano_x,Cladocopium_leituras,Durusdinium_leituras,...,q99_sst,Numero_do_tubo,Latitude_y,Longitude_y,Profundidade_metros_y,Dia_y,Mes_y,Ano_y,Bleaching,Death
32,80,ACR_408,-12.911220,40.499950,2.0,14,5,2024,1799922,60687,...,30.488900,408,12.911220,40.499950,2.0,14,5,2024,0.0,0.0
33,81,ACR_412,-12.911220,40.499950,2.9,14,5,2024,2257642,51048,...,30.488900,412,12.911220,40.499950,2.9,14,5,2024,1.0,1.0
34,82,ACR_414,-12.964526,40.548636,3.7,13,5,2024,2413318,67382,...,30.439664,414,-12.964526,40.548636,3.7,13,5,2024,0.0,0.0
35,83,ACR_416,-12.964526,40.548636,5.4,13,5,2024,1109946,56351,...,30.439664,416,-12.964526,40.548636,5.4,13,5,2024,0.0,0.0
36,84,ACR_419,-12.964526,40.548636,4.2,13,5,2024,5730941,91468,...,30.439664,419,-12.964526,40.548636,4.2,13,5,2024,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,231,ACR_723,-12.905278,40.498611,2.9,3,7,2024,1080713,100444,...,30.510966,723,-12.905278,40.498611,2.9,3,7,2024,1.0,0.0
163,232,ACR_726,-12.905278,40.498611,3.0,3,7,2024,1039092,73789,...,30.510966,726,-12.905278,40.498611,3.0,3,7,2024,1.0,1.0
164,233,ACR_729,-12.905278,40.498611,2.6,3,7,2024,671937,32905,...,30.510966,729,-12.905278,40.498611,2.6,3,7,2024,1.0,0.0
165,234,ACR_801,-12.915556,40.500278,1.8,8,7,2024,205679,12452,...,28.263662,801,-12.915556,40.500278,1.8,8,7,2024,0.0,0.0


In [73]:
x=meta_bleaching24.loc[:,['mean_sst']]
x

,mean_sst
32,28.251490
33,28.251490
34,28.202414
35,28.202414
36,28.202414
...,...
162,28.254794
163,28.254794
164,28.254794
165,28.263662


In [89]:
x=meta_bleaching24.loc[:,['mean_sst']]
y=meta_bleaching24.loc[:,['Bleaching']].values.ravel()
mod = sklearn.linear_model.LogisticRegression(max_iter=1000)
mod.fit(x,y)
mod_preds = mod.predict(x)
r2 = sklearn.metrics.r2_score(y,mod_preds)
r2

-0.306930693069307

In [81]:
x=meta_bleaching24.loc[:,['q99_sst']]
y=meta_bleaching24.loc[:,['Bleaching']].values.ravel()
mod = sklearn.linear_model.LogisticRegression(max_iter=1000)
mod.fit(x,y)
mod_preds = mod.predict(pd.DataFrame(x))
r2 = sklearn.metrics.r2_score(y,mod_preds)
r2

-0.306930693069307

In [88]:
meta_bleaching24.to_csv('bleaching_meta_sst.csv')